In [ ]:
###--- Two-way ANOVA with homogeneial data ---###

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols


harvest = pd.read_csv("HarvestData.csv")

# extract the traits to run ANOVA  (count from column)

traits = harvest.columns[5:]
anova_results = []

# Two-way ANOVA
for trait in traits:
    try:
        # Define the model: Genotype * Treatment (Interaction)
        formula = f"{trait} ~ C(Genotype) * C(Treatment)"
        model = ols(formula, data=harvest).fit()
        
        anova_table = sm.stats.anova_lm(model, typ=2)
        
        # p-value
        p_genotype = anova_table.loc['C(Genotype)', 'PR(>F)']
        p_treatment = anova_table.loc['C(Treatment)', 'PR(>F)']
        p_interaction = anova_table.loc['C(Genotype):C(Treatment)', 'PR(>F)']
        
        anova_results.append({
            "Trait": trait,
            "p_Genotype": p_genotype,
            "p_Treatment": p_treatment,
            "p_Genotype_Treatment": p_interaction
        })
    except Exception as e:
        print(f"Error processing {trait}: {e}")


output_df = pd.DataFrame(anova_results)
output_df[['p_Genotype', 'p_Treatment', 'p_Genotype_Treatment']] = \
    output_df[['p_Genotype', 'p_Treatment', 'p_Genotype_Treatment']].apply(lambda x: x.apply(lambda y: f"{y:.3g}"))

output_df = output_df.sort_values(by='p_Treatment')
output_df.to_csv("Results/Table1_python.txt", sep="\t", index=False)

print(output_df.head())

In [ ]:
###--- Two-way ANOVA combined with assumption test (Log transformation) ---###
###### take only part of log transformation #######

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import shapiro, levene
import os

file_path = "HarvestData.csv"
harvest = pd.read_csv(file_path)
traits = harvest.columns[5:]
results = []

for trait in traits:
    try:
        df_clean = harvest[[trait, 'Genotype', 'Treatment']].dropna()
        
        # 1. Fit the basic ANOVA model
        formula = f"{trait} ~ C(Genotype) * C(Treatment)"
        model = ols(formula, data=df_clean).fit()
        
        # 2. Extract residuals and test normality (Shapiro-Wilk test)
        residuals = model.resid
        shapiro_p = shapiro(residuals)[1]
        
        # 3. Homogeneity test: Levene's test (prior assumption of two-way ANOVA)
        groups = [group[trait].values for name, group in df_clean.groupby(['Genotype', 'Treatment'])]
        levene_p = levene(*groups)[1]
        
        # 4. Data transformation
        status = "Pass"
        if shapiro_p < 0.05 or levene_p < 0.05:
            status = "Need Transformation"

            if (df_clean[trait] > 0).all():
                df_clean[f"log_{trait}"] = np.log10(df_clean[trait])
                log_formula = f"log_{trait} ~ C(Genotype) * C(Treatment)"
                model = ols(log_formula, data=df_clean).fit()
                # re-test after transformation
                shapiro_p = shapiro(model.resid)[1]
                status = "Transformed(Log10)"

        # 5. Extract ANOVA result in table
        anova_table = sm.stats.anova_lm(model, typ=2)
        
        results.append({
            "Trait": trait,
            "Status": status,
            "Residual_Normality_p": shapiro_p,
            "Homogeneity_p": levene_p,
            "p_Genotype": anova_table.loc['C(Genotype)', 'PR(>F)'],
            "p_Treatment": anova_table.loc['C(Treatment)', 'PR(>F)'],
            "p_GxE": anova_table.loc['C(Genotype):C(Treatment)', 'PR(>F)']
        })
        
    except Exception as e:
        print(f"Error in {trait}: {e}")


final_df = pd.DataFrame(results)
os.makedirs("Results", exist_ok=True)
final_df.to_csv("Results/Final_ANOVA_Diagnostics.csv", sep="\t", index=False)
print(final_df.head())

In [ ]:
###--- Interaction table and Tukey HSD ---###

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import os

base_dir = "./"
data_path = os.path.join(base_dir, "HarvestData.csv")
diag_path = os.path.join(base_dir, "Results/Final_ANOVA_Diagnostics.csv")

harvest = pd.read_csv(data_path)
diag_results = pd.read_csv(diag_path, sep="\t")

# Extract only significant GxE traits
gxe_traits = diag_results[diag_results['p_GxE'] < 0.05]

plot_dir = os.path.join(base_dir, "Results/Plots")
os.makedirs(plot_dir, exist_ok=True)

tukey_summary = []

# Analysis for each trait loop
for _, row in gxe_traits.iterrows():
    trait = row['Trait']
    status = row['Status']
    
    df_sub = harvest[['Genotype', 'Treatment', trait]].dropna()
    
# Log transformation
    if "Log10" in status:
        df_sub[trait] = np.log10(df_sub[trait])
        ylabel = f"log10({trait})"
    else:
        ylabel = trait

    # --- 1. Interaction Plot ---
    plt.figure(figsize=(10, 6))
    sns.pointplot(data=df_sub, x='Treatment', y=trait, hue='Genotype', 
                  capsize=.1, markers="o", linestyles="-")
    plt.title(f"Interaction Plot: {trait} (GxE p={row['p_GxE']:.2e})")
    plt.ylabel(ylabel)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.savefig(os.path.join(plot_dir, f"Interaction_{trait}.png"), dpi=300)
    plt.close()

    # --- 2. Tukey HSD ---
    df_sub['Group'] = df_sub['Genotype'].astype(str) + "_" + df_sub['Treatment'].astype(str)
    
    tukey = pairwise_tukeyhsd(endog=df_sub[trait], groups=df_sub['Group'], alpha=0.05)
    
    tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
    tukey_df['Trait'] = trait
    tukey_summary.append(tukey_df)

if tukey_summary:
    final_tukey_df = pd.concat(tukey_summary)
    final_tukey_df.to_csv(os.path.join(base_dir, "Results/Tukey_HSD_Results.txt"), sep="\t", index=False)
    print(f"{len(gxe_traits)} traits are analyzed.")

분석 완료! 30개 형질의 시각화 및 사후검정이 완료되었습니다.
